# ko → num (김서현)

`baseline.ipynb`를 복사해 ko→num 방향으로 바꾼 노트북. 모델 구조와 공통 함수(이름·입출력·저장 형식)는 베이스라인 그대로이고, 바꾼 것은 학습 데이터 구성뿐이다.

**직접 구현: 짧은 숫자(1~3자리) 오버샘플링**

- 왜: 데이터가 5자리 이상은 자릿수별 15,000개씩인데 1자리 10개, 2자리 90개, 3자리 900개뿐이라, 원본 그대로 학습하면 짧은 숫자를 거의 못 본다 (베이스라인 사전 실행에서는 1~2자리 오답이 많았다).
- 무엇을: train에 있는 1~3자리 쌍을 자릿수별로 `OS_TARGET`개가 될 때까지 반복해서 `make_loaders(..., extra_train=...)`로 추가한다. valid/test는 원본 그대로 쓴다.
- 비교: 원본 데이터로 학습한 모델(실험 A) vs 오버샘플링 모델(실험 B)의 test 자릿수별 정확도.
- 숫자는 `int` 변환 없이 문자열로만 다룬다. 자릿수도 문자열 길이(`len`)로 센다.

데이터
- `num2kor.txt`: 한글 (CRLF), 입력은 글자 단위(공백 포함), 출력은 숫자 글자 단위
- train/valid/test 8:1:1, 숫자 해시 기준 (num→ko와 split 동일)

실행
- Colab GPU로 전체 실행하면 `project01/ko2num.pt` 생성

In [1]:
# 설정
SRC, TGT = 'ko', 'num'  # num, en, ko

DATA_DIR = '../data'
SAVE_DIR = '..'  # pt 저장 위치

CONFIG = dict(
    emb_dim=128,
    hid_dim=256,
    n_layers=2,
    dropout=0.2,
    rnn='gru',  # gru / lstm
    attention='luong',  # 'luong' / None
)
BATCH_SIZE = 128
EPOCHS = 15
LR = 1e-3
TEACHER_FORCING = 0.5
BEAM_SIZE = 5
SEED = 42
MAX_TRAIN = None  # 디버깅용 train 샘플 수

# [직접 구현] 짧은 숫자 오버샘플링 설정
OS_MAX_DIGITS = 3   # 이 자릿수 이하를 "짧은 숫자"로 본다
OS_TARGET = 3000    # 짧은 자릿수마다 train에서 이 개수가 되도록 반복

In [2]:
# Colab 사용 시: 데이터가 없으면 레포를 클론하고 work 폴더로 이동
import os
if not os.path.exists(os.path.join(DATA_DIR, 'num2kor.txt')):
    if not os.path.exists('2026-nlp'):
        !git clone https://github.com/jungyy808/2026-nlp.git
    %cd 2026-nlp/project01/work

/content/2026-nlp/project01/work


In [3]:
import os, random, time, zlib, math
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def set_seed(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cuda


## 데이터

In [4]:
FILES = {'en': 'num2txt.txt', 'ko': 'num2kor.txt'}
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']


def tokenize(text, lang):
    # en: 단어 단위 / num, ko: 문자 단위 (공백 포함)
    return text.split() if lang == 'en' else list(text)


def detokenize(tokens, lang):
    return ' '.join(tokens) if lang == 'en' else ''.join(tokens)


def load_pairs(path):
    # CRLF 제거
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\r\n')
            if not line:
                continue
            num, text = line.split('\t', 1)
            pairs.append((num.strip(), text.strip()))
    return pairs


def get_pairs(src_lang, tgt_lang, data_dir=DATA_DIR):
    # (src, tgt, key) 반환, key는 split 기준
    assert src_lang != tgt_lang and {src_lang, tgt_lang} <= {'num', 'en', 'ko'}
    tables = {lang: dict(load_pairs(os.path.join(data_dir, FILES[lang])))
              for lang in (src_lang, tgt_lang) if lang != 'num'}
    if 'num' in (src_lang, tgt_lang):
        other = tgt_lang if src_lang == 'num' else src_lang
        keys = list(tables[other])
    else:  # en-ko: 공통 key만 사용
        keys = sorted(set(tables['en']) & set(tables['ko']))

    def text(lang, k):
        return k if lang == 'num' else tables[lang][k]

    return [(text(src_lang, k), text(tgt_lang, k), k) for k in keys]


def split_of(key):
    # key 해시 기준 8:1:1 분할
    b = zlib.crc32(key.encode('utf-8')) % 10
    return 'test' if b == 0 else 'valid' if b == 1 else 'train'


class Vocab:
    def __init__(self, lang, sentences=()):
        self.lang = lang
        tokens = sorted({t for s in sentences for t in tokenize(s, lang)})
        self.itos = SPECIALS + tokens
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    @classmethod
    def from_itos(cls, lang, itos):
        v = cls(lang)
        v.itos = list(itos)
        v.stoi = {t: i for i, t in enumerate(v.itos)}
        return v

    def encode(self, text):  # sos/eos 제외
        return [self.stoi.get(t, UNK) for t in tokenize(text, self.lang)]

    def decode(self, ids):
        out = []
        for i in ids:
            i = int(i)
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return detokenize(out, self.lang)

    def __len__(self):
        return len(self.itos)


class PairDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src = [src_vocab.encode(s) for s, _, _ in pairs]
        self.tgt = [[SOS] + tgt_vocab.encode(t) + [EOS] for _, t, _ in pairs]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        return self.src[i], self.tgt[i]


def collate(batch):
    srcs, tgts = zip(*batch)
    src_len = torch.tensor([len(s) for s in srcs])
    src = torch.full((len(srcs), max(src_len)), PAD, dtype=torch.long)
    tgt = torch.full((len(tgts), max(len(t) for t in tgts)), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src[i, :len(s)] = torch.tensor(s)
        tgt[i, :len(t)] = torch.tensor(t)
    return src, src_len, tgt


def make_loaders(pairs, src_lang, tgt_lang, batch_size=BATCH_SIZE, max_train=None, extra_train=None):
    # extra_train: 증강 등으로 추가할 (src, tgt, key) 리스트. train에만 들어가고 valid/test는 원본만 사용
    src_vocab = Vocab(src_lang, [s for s, _, _ in pairs])
    tgt_vocab = Vocab(tgt_lang, [t for _, t, _ in pairs])
    parts = defaultdict(list)
    for p in pairs:
        parts[split_of(p[2])].append(p)
    if max_train:
        parts['train'] = random.Random(SEED).sample(parts['train'], min(max_train, len(parts['train'])))
    if extra_train:
        held_out = {p[2] for p in parts['valid'] + parts['test']}
        extra = [p for p in extra_train if p[2] not in held_out]  # valid/test 숫자와 겹치는 건 제외
        print(f'extra_train: {len(extra):,}개 추가 (valid/test와 겹쳐서 제외 {len(extra_train) - len(extra):,}개)')
        parts['train'] = parts['train'] + extra
    loaders = [DataLoader(PairDataset(parts[name], src_vocab, tgt_vocab), batch_size=batch_size,
                          shuffle=(name == 'train'), collate_fn=collate)
               for name in ('train', 'valid', 'test')]
    return (*loaders, src_vocab, tgt_vocab)

In [5]:
print('작업 위치:', os.getcwd())
for f in FILES.values():
    assert os.path.exists(os.path.join(DATA_DIR, f)), f'{os.path.abspath(os.path.join(DATA_DIR, f))} 없음 (Colab이면 clone 셀 확인)'
print('pt 저장 위치:', os.path.abspath(SAVE_DIR))

pairs = get_pairs(SRC, TGT)
train_loader, valid_loader, test_loader, src_vocab, tgt_vocab = make_loaders(
    pairs, SRC, TGT, BATCH_SIZE, MAX_TRAIN)

MAX_LEN = max(len(tokenize(t, TGT)) for _, t, _ in pairs) + 10  # 데이터 기준 최대 길이
print(f'{SRC} -> {TGT}: 전체 {len(pairs):,}쌍 | train {len(train_loader.dataset):,} / '
      f'valid {len(valid_loader.dataset):,} / test {len(test_loader.dataset):,}')
print(f'vocab: src {len(src_vocab)}, tgt {len(tgt_vocab)} | 디코딩 최대 길이 {MAX_LEN}')
for s, t, _ in pairs[:3]:
    print(f'  {s!r} -> {t!r}')

작업 위치: /content/2026-nlp/project01/work
pt 저장 위치: /content/2026-nlp/project01
ko -> num: 전체 100,000쌍 | train 80,321 / valid 9,718 / test 9,961
vocab: src 20, tgt 14 | 디코딩 최대 길이 20
  '이만 천오백사십육' -> '21546'
  '이만 육천칠백삼십오' -> '26735'
  '오백십만 칠천사백팔십사' -> '5107484'


## 모델 (GRU/LSTM encoder-decoder + attention)

In [6]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = RNN(emb_dim, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_len):
        x = self.drop(self.emb(src))
        packed = nn.utils.rnn.pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        return out, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention):
        super().__init__()
        assert attention in ('luong', None), f'지원하지 않는 attention: {attention}'  # 옵션 추가 시 여기에도 추가
        assert rnn in ('gru', 'lstm'), f'지원하지 않는 rnn: {rnn}'
        self.attention = attention
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = RNN(emb_dim, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0)
        if attention == 'luong':
            self.attn = nn.Linear(hid_dim, hid_dim, bias=False)  # Luong (general)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        self.out = nn.Linear(hid_dim, vocab_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, prev_token, hidden, enc_out, src_mask):
        x = self.drop(self.emb(prev_token)).unsqueeze(1)
        h, hidden = self.rnn(x, hidden)
        h = h.squeeze(1)
        if self.attention == 'luong':
            scores = torch.bmm(enc_out, self.attn(h).unsqueeze(2)).squeeze(2)
            scores = scores.masked_fill(~src_mask, -1e9)
            ctx = torch.bmm(F.softmax(scores, dim=1).unsqueeze(1), enc_out).squeeze(1)
            h = torch.tanh(self.combine(torch.cat([h, ctx], dim=1)))
        return self.out(self.drop(h)), hidden


class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, emb_dim=128, hid_dim=256,
                 n_layers=2, dropout=0.2, rnn='gru', attention='luong'):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn)
        self.decoder = Decoder(tgt_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention)

    def encode(self, src, src_len):
        enc_out, hidden = self.encoder(src, src_len)
        return (hidden, enc_out, src != PAD)  # (hidden, enc_out, mask)

    def decode_step(self, prev_token, state):
        hidden, enc_out, mask = state
        logits, hidden = self.decoder(prev_token, hidden, enc_out, mask)
        return logits, (hidden, enc_out, mask)

    def forward(self, src, src_len, tgt, teacher_forcing=0.5):
        # tgt[:, 1:] 예측
        state = self.encode(src, src_len)
        token, outputs = tgt[:, 0], []
        for t in range(1, tgt.size(1)):
            logits, state = self.decode_step(token, state)
            outputs.append(logits)
            use_tf = random.random() < teacher_forcing
            token = tgt[:, t] if use_tf else logits.argmax(1)
        return torch.stack(outputs, dim=1)


def reorder_state(state, idx):
    # beam search용 배치 재정렬
    hidden, enc_out, mask = state
    if isinstance(hidden, tuple):
        hidden = tuple(h.index_select(1, idx) for h in hidden)
    else:
        hidden = hidden.index_select(1, idx)
    return hidden, enc_out.index_select(0, idx), mask.index_select(0, idx)


model = Seq2Seq(len(src_vocab), len(tgt_vocab), **CONFIG).to(DEVICE)
print('params:', sum(p.numel() for p in model.parameters()))

params: 1587214


## 학습, greedy / beam 디코딩

In [7]:
@torch.no_grad()
def greedy_batch(model, src, src_len, max_len=None):
    # batch greedy
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)  # 로드한 모델은 저장된 max_len 사용
    model.eval()
    state = model.encode(src, src_len)
    token = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    out = []
    for _ in range(max_len):
        logits, state = model.decode_step(token, state)
        token = logits.argmax(1)
        out.append(token)
        done |= token == EOS
        if done.all():
            break
    return torch.stack(out, dim=1)


def _to_tensor(text, vocab):
    ids = vocab.encode(text)
    return torch.tensor([ids], device=DEVICE), torch.tensor([len(ids)])


def greedy_decode(model, src_text, src_vocab, tgt_vocab, max_len=None):
    src, src_len = _to_tensor(src_text, src_vocab)
    return tgt_vocab.decode(greedy_batch(model, src, src_len, max_len)[0].tolist())


@torch.no_grad()
def beam_decode(model, src_text, src_vocab, tgt_vocab, beam=BEAM_SIZE, max_len=None):
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)
    model.eval()
    src, src_len = _to_tensor(src_text, src_vocab)
    state = model.encode(src, src_len)
    beams = [([], 0.0)]  # (seq, log prob)
    finished = []
    for _ in range(max_len):
        tokens = torch.tensor([b[0][-1] if b[0] else SOS for b in beams], device=DEVICE)
        if len(beams) != state[1].size(0):
            state = reorder_state(state, torch.zeros(len(beams), dtype=torch.long, device=DEVICE))
        logits, state = model.decode_step(tokens, state)
        logp = F.log_softmax(logits, dim=1)
        cand = []
        for i, (seq, score) in enumerate(beams):
            top = logp[i].topk(beam)
            for lp, tok in zip(top.values.tolist(), top.indices.tolist()):
                cand.append((seq + [tok], score + lp, i))
        cand.sort(key=lambda c: c[1], reverse=True)
        beams, keep = [], []
        for seq, score, i in cand:
            if seq[-1] == EOS:
                finished.append((seq, score / len(seq)))  # 길이 정규화
            else:
                beams.append((seq, score)); keep.append(i)
            if len(beams) == beam:
                break
        if not beams or len(finished) >= beam:
            break
        state = reorder_state(state, torch.tensor(keep, device=DEVICE))
    if not finished:
        finished = [(seq, score / len(seq)) for seq, score in beams]
    best = max(finished, key=lambda f: f[1])[0]
    return tgt_vocab.decode(best)


@torch.no_grad()
def evaluate(model, loader, tgt_vocab):
    # exact match 기준
    model.eval()
    total_loss, n_tok, correct, n = 0.0, 0, 0, 0
    for src, src_len, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        logits = model(src, src_len, tgt, teacher_forcing=1.0)
        gold = tgt[:, 1:]
        total_loss += F.cross_entropy(logits.reshape(-1, logits.size(-1)), gold.reshape(-1),
                                      ignore_index=PAD, reduction='sum').item()
        n_tok += (gold != PAD).sum().item()
        pred = greedy_batch(model, src, src_len)
        for p, g in zip(pred.tolist(), gold.tolist()):
            correct += tgt_vocab.decode(p) == tgt_vocab.decode(g)
            n += 1
    return {'loss': total_loss / n_tok, 'exact_match': correct / n}


def train(model, train_loader, valid_loader, epochs=EPOCHS, lr=LR, save_path=None):
    # best valid acc 모델 복원
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history, best, best_state = [], -1, None
    for ep in range(1, epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        for src, src_len, tgt in train_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            logits = model(src, src_len, tgt, TEACHER_FORCING)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
        val = evaluate(model, valid_loader, tgt_vocab)
        history.append({'epoch': ep, 'train_loss': total / len(train_loader), **{f'valid_{k}': v for k, v in val.items()}})
        print(f"epoch {ep:2d} | train loss {history[-1]['train_loss']:.4f} | valid loss {val['loss']:.4f} "
              f"| valid acc {val['exact_match']*100:.2f}% | {time.time()-t0:.0f}s")
        if val['exact_match'] > best:
            best, best_state = val['exact_match'], {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'best valid acc: {best*100:.2f}%')
    return history

In [8]:
history = train(model, train_loader, valid_loader, EPOCHS, LR)

epoch  1 | train loss 0.3210 | valid loss 0.0226 | valid acc 95.58% | 37s
epoch  2 | train loss 0.0163 | valid loss 0.0011 | valid acc 99.88% | 34s
epoch  3 | train loss 0.0089 | valid loss 0.0006 | valid acc 99.95% | 34s
epoch  4 | train loss 0.0042 | valid loss 0.0002 | valid acc 99.99% | 34s
epoch  5 | train loss 0.0062 | valid loss 0.0097 | valid acc 99.17% | 34s
epoch  6 | train loss 0.0044 | valid loss 0.0019 | valid acc 99.90% | 33s
epoch  7 | train loss 0.0022 | valid loss 0.0032 | valid acc 99.27% | 34s
epoch  8 | train loss 0.0053 | valid loss 0.0005 | valid acc 99.96% | 34s
epoch  9 | train loss 0.0015 | valid loss 0.0006 | valid acc 99.95% | 34s
epoch 10 | train loss 0.0031 | valid loss 0.0017 | valid acc 99.78% | 34s
epoch 11 | train loss 0.0024 | valid loss 0.0004 | valid acc 99.97% | 34s
epoch 12 | train loss 0.0015 | valid loss 0.0036 | valid acc 99.75% | 34s
epoch 13 | train loss 0.0027 | valid loss 0.0001 | valid acc 99.98% | 35s
epoch 14 | train loss 0.0002 | valid l

## test 평가

In [9]:
@torch.no_grad()
def predict_all(model, loader, tgt_vocab):
    rows, i = [], 0
    for src, src_len, _ in loader:
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        for p in pred.tolist():
            s, t, k = loader.dataset.pairs[i]
            rows.append((s, t, tgt_vocab.decode(p), k)); i += 1
    return rows


def exact_match(rows):
    return sum(r[1] == r[2] for r in rows) / len(rows)


def acc_by_length(rows):
    stat = defaultdict(lambda: [0, 0])
    for _, gold, pred, key in rows:
        stat[len(key)][0] += gold == pred
        stat[len(key)][1] += 1
    print('자릿수 | 개수 | 정확도')
    for L in sorted(stat):
        c, n = stat[L]
        print(f'{L:6d} | {n:6d} | {c/n*100:6.2f}%')


def show_examples(rows, n=5):
    right = [r for r in rows if r[1] == r[2]][:n]
    wrong = [r for r in rows if r[1] != r[2]][:n]
    for title, group in (('맞은 예시', right), ('틀린 예시', wrong)):
        print(f'--- {title} ---')
        for s, gold, pred, _ in group:
            print(f'입력: {s}\n정답: {gold}\n예측: {pred}\n')


test_rows = predict_all(model, test_loader, tgt_vocab)
print(f'{SRC} -> {TGT} test exact match (greedy): {exact_match(test_rows)*100:.2f}%  (n={len(test_rows):,})\n')
acc_by_length(test_rows)
print()
show_examples(test_rows)

ko -> num test exact match (greedy): 99.99%  (n=9,961)

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |   1491 | 100.00%
     6 |   1497 | 100.00%
     7 |   1508 | 100.00%
     8 |   1493 | 100.00%
     9 |   1474 | 100.00%
    10 |   1502 |  99.93%

--- 맞은 예시 ---
입력: 구억 천삼백이십육만 구천사백삼십삼
정답: 913269433
예측: 913269433

입력: 육천이백십구만 팔천팔백사십이
정답: 62198842
예측: 62198842

입력: 십사만 사천칠십사
정답: 144074
예측: 144074

입력: 삼백십육만 천팔백오십육
정답: 3161856
예측: 3161856

입력: 구천이백십오만 팔백사십일
정답: 92150841
예측: 92150841

--- 틀린 예시 ---
입력: 육십팔억 구천칠백사십이만
정답: 6897420000
예측: 689742000



In [10]:
# greedy vs beam (test 200개)
sample = test_loader.dataset.pairs[:200]
g = sum(greedy_decode(model, s, src_vocab, tgt_vocab) == t for s, t, _ in sample)
b = sum(beam_decode(model, s, src_vocab, tgt_vocab, BEAM_SIZE) == t for s, t, _ in sample)
print(f'test 200개: greedy {g/len(sample)*100:.1f}% | beam({BEAM_SIZE}) {b/len(sample)*100:.1f}%')

test 200개: greedy 100.0% | beam(5) 100.0%


## 직접 구현: 짧은 숫자(1~3자리) 오버샘플링 (김서현)

위까지가 실험 A(원본 데이터)다. 여기부터 실험 B로, train의 1~3자리 쌍을 반복해서 늘린 뒤 같은 설정으로 새 모델을 학습하고 자릿수별 정확도를 비교한다.

In [11]:
# [직접 구현] 짧은 숫자(1~3자리) 오버샘플링
# 왜 바꿨나: 5자리 이상은 자릿수별 15,000개인데 1자리 10개, 2자리 90개, 3자리 900개뿐이다.
#            원본 train에서는 짧은 숫자가 전체의 1% 정도라 모델이 "칠", "삼십이" 같은 짧은 입력을
#            거의 못 보고, 긴 숫자 패턴에 맞춰져서 짧은 숫자를 틀릴 수 있다.
# 무엇을 바꿨나: train split의 1~3자리 쌍을 자릿수별로 OS_TARGET개가 될 때까지 반복해서 추가한다.
#                새 데이터를 만드는 게 아니라 있는 쌍을 여러 번 보여 주는 것이다.
# 주의: 자릿수는 key(숫자 문자열)의 len()으로 센다. int 변환은 쓰지 않는다.
#       valid/test는 건드리지 않는다 (make_loaders가 extra_train을 train에만 넣는다).

def digit_counts(pair_list):
    # 자릿수별 개수
    cnt = defaultdict(int)
    for _, _, key in pair_list:
        cnt[len(key)] += 1
    return dict(sorted(cnt.items()))


def oversample_short(train_pairs, max_digits=OS_MAX_DIGITS, target=OS_TARGET):
    # 추가로 넣을 쌍(복제본)만 반환한다. 원본은 이미 train에 있으므로 포함하지 않는다.
    groups = defaultdict(list)
    for p in train_pairs:
        if len(p[2]) <= max_digits:
            groups[len(p[2])].append(p)
    extra = []
    for L, group in sorted(groups.items()):
        need = target - len(group)
        if need <= 0:
            continue
        reps, rem = divmod(need, len(group))
        # 전체를 reps번 반복하고, 남는 개수는 무작위로 골라 채운다
        extra += group * reps + random.Random(SEED + L).sample(group, rem)
    return extra


train_pairs = train_loader.dataset.pairs  # 실험 A가 쓴 원본 train
extra = oversample_short(train_pairs)

before = digit_counts(train_pairs)
after = digit_counts(train_pairs + extra)
print('자릿수 | 원본 train | 오버샘플링 후')
for L in before:
    print(f'{L:6d} | {before[L]:10,d} | {after[L]:10,d}')
print(f'추가된 쌍: {len(extra):,}개 (train {len(train_pairs):,} -> {len(train_pairs) + len(extra):,})')

자릿수 | 원본 train | 오버샘플링 후
     1 |          7 |      3,000
     2 |         67 |      3,000
     3 |        732 |      3,000
     4 |      7,177 |      7,177
     5 |     12,025 |     12,025
     6 |     12,061 |     12,061
     7 |     12,055 |     12,055
     8 |     12,127 |     12,127
     9 |     12,035 |     12,035
    10 |     12,035 |     12,035
추가된 쌍: 8,194개 (train 80,321 -> 88,515)


In [12]:
# 실험 B: 오버샘플링한 train으로 새 모델 학습 (모델 구조·하이퍼파라미터는 실험 A와 동일)
set_seed(SEED)
train_loader_os, _, _, src_vocab_os, tgt_vocab_os = make_loaders(
    pairs, SRC, TGT, BATCH_SIZE, MAX_TRAIN, extra_train=extra)
# vocab은 전체 pairs로 만들기 때문에 실험 A와 같아야 한다
assert src_vocab_os.itos == src_vocab.itos and tgt_vocab_os.itos == tgt_vocab.itos

model_os = Seq2Seq(len(src_vocab), len(tgt_vocab), **CONFIG).to(DEVICE)
history_os = train(model_os, train_loader_os, valid_loader, EPOCHS, LR)  # valid는 원본 그대로

extra_train: 8,194개 추가 (valid/test와 겹쳐서 제외 0개)
epoch  1 | train loss 0.2946 | valid loss 0.0714 | valid acc 88.93% | 39s
epoch  2 | train loss 0.0143 | valid loss 0.0010 | valid acc 99.93% | 39s
epoch  3 | train loss 0.0099 | valid loss 0.0040 | valid acc 99.59% | 39s
epoch  4 | train loss 0.0035 | valid loss 0.0002 | valid acc 99.98% | 41s
epoch  5 | train loss 0.0094 | valid loss 0.0030 | valid acc 99.65% | 40s
epoch  6 | train loss 0.0053 | valid loss 0.0013 | valid acc 99.91% | 39s
epoch  7 | train loss 0.0036 | valid loss 0.0005 | valid acc 99.92% | 38s
epoch  8 | train loss 0.0020 | valid loss 0.0001 | valid acc 99.99% | 39s
epoch  9 | train loss 0.0002 | valid loss 0.0001 | valid acc 99.99% | 39s
epoch 10 | train loss 0.0001 | valid loss 0.0001 | valid acc 99.99% | 38s
epoch 11 | train loss 0.0001 | valid loss 0.0001 | valid acc 99.99% | 38s
epoch 12 | train loss 0.0006 | valid loss 0.0012 | valid acc 99.77% | 38s
epoch 13 | train loss 0.0106 | valid loss 0.0013 | valid acc 99.9

In [13]:
# 비교: 원본(A) vs 오버샘플링(B), test 자릿수별 exact match
test_rows_os = predict_all(model_os, test_loader, tgt_vocab)  # test는 원본만


def length_stats(rows):
    stat = defaultdict(lambda: [0, 0])  # 자릿수 -> [맞은 수, 전체]
    for _, gold, pred, key in rows:
        stat[len(key)][0] += gold == pred
        stat[len(key)][1] += 1
    return stat


def pct(c, n):
    return f'{c / n * 100:6.2f}%' if n else '   -   '


stat_a, stat_b = length_stats(test_rows), length_stats(test_rows_os)
print('자릿수 |  개수  | A 원본  | B 오버샘플링')
for L in sorted(stat_a):
    print(f'{L:6d} | {stat_a[L][1]:6d} | {pct(*stat_a[L])} | {pct(*stat_b[L])}')

short = [L for L in stat_a if L <= OS_MAX_DIGITS]
long_ = [L for L in stat_a if L > OS_MAX_DIGITS]
for name, Ls in ((f'1~{OS_MAX_DIGITS}자리', short), (f'{OS_MAX_DIGITS + 1}자리 이상', long_)):
    n = sum(stat_a[L][1] for L in Ls)
    print(f'{name:>9} | {n:6d} | {pct(sum(stat_a[L][0] for L in Ls), n)} | {pct(sum(stat_b[L][0] for L in Ls), n)}')
print(f'     전체 | {len(test_rows):6d} | {exact_match(test_rows) * 100:6.2f}% | {exact_match(test_rows_os) * 100:6.2f}%')

# 짧은 숫자 중 두 모델의 정답 여부가 갈린 예시
print('\n--- 1~3자리에서 A와 B의 결과가 다른 예시 ---')
shown = 0
for (s, gold, pa, key), (_, _, pb, _) in zip(test_rows, test_rows_os):
    if len(key) <= OS_MAX_DIGITS and (pa == gold) != (pb == gold):
        print(f'입력: {s!r} | 정답: {gold} | A: {pa} | B: {pb}')
        shown += 1
        if shown == 10:
            break
if not shown:
    print('(없음)')

print('\n--- B 모델 예시 ---')
show_examples(test_rows_os)

자릿수 |  개수  | A 원본  | B 오버샘플링
     1 |      1 | 100.00% | 100.00%
     2 |      8 | 100.00% | 100.00%
     3 |     74 | 100.00% | 100.00%
     4 |    913 | 100.00% | 100.00%
     5 |   1491 | 100.00% | 100.00%
     6 |   1497 | 100.00% | 100.00%
     7 |   1508 | 100.00% | 100.00%
     8 |   1493 | 100.00% | 100.00%
     9 |   1474 | 100.00% | 100.00%
    10 |   1502 |  99.93% | 100.00%
    1~3자리 |     83 | 100.00% | 100.00%
   4자리 이상 |   9878 |  99.99% | 100.00%
     전체 |   9961 |  99.99% | 100.00%

--- 1~3자리에서 A와 B의 결과가 다른 예시 ---
(없음)

--- B 모델 예시 ---
--- 맞은 예시 ---
입력: 구억 천삼백이십육만 구천사백삼십삼
정답: 913269433
예측: 913269433

입력: 육천이백십구만 팔천팔백사십이
정답: 62198842
예측: 62198842

입력: 십사만 사천칠십사
정답: 144074
예측: 144074

입력: 삼백십육만 천팔백오십육
정답: 3161856
예측: 3161856

입력: 구천이백십오만 팔백사십일
정답: 92150841
예측: 92150841

--- 틀린 예시 ---


In [14]:
# 제출할 모델 선택: test로 고르면 test 점수가 부풀려지므로 valid 정확도로 고른다.
# 같으면 짧은 숫자를 더 본 오버샘플링 모델을 쓴다.
valid_a = max(h['valid_exact_match'] for h in history)
valid_b = max(h['valid_exact_match'] for h in history_os)
USE_OS = valid_b >= valid_a
final_model = model_os if USE_OS else model
final_rows = test_rows_os if USE_OS else test_rows
final_history = history_os if USE_OS else history
print(f'valid acc  A 원본 {valid_a * 100:.2f}% | B 오버샘플링 {valid_b * 100:.2f}%')
print(f"제출 모델: {'B 오버샘플링' if USE_OS else 'A 원본'} (test {exact_match(final_rows) * 100:.2f}%)")

# 제출 모델의 greedy vs beam (test 200개)
g = sum(greedy_decode(final_model, s, src_vocab, tgt_vocab) == t for s, t, _ in sample)
b = sum(beam_decode(final_model, s, src_vocab, tgt_vocab, BEAM_SIZE) == t for s, t, _ in sample)
print(f'test 200개: greedy {g / len(sample) * 100:.1f}% | beam({BEAM_SIZE}) {b / len(sample) * 100:.1f}%')

valid acc  A 원본 99.99% | B 오버샘플링 99.99%
제출 모델: B 오버샘플링 (test 100.00%)
test 200개: greedy 100.0% | beam(5) 100.0%


### 결과 (test 9,961개, exact match)

| | A 원본 | B 오버샘플링 |
|---|---|---|
| 1~3자리 (83개) | 100.00% | 100.00% |
| 4자리 이상 (9,878개) | 99.99% | 100.00% |
| 전체 | 99.99% | 100.00% |

- ko→num은 원본 데이터만으로도 15 epoch 학습하면 1~3자리를 전부 맞혔다. 그래서 짧은 숫자에서는 오버샘플링 효과가 수치로 드러나지 않았다.
- 전체 차이는 10자리 1개(`육십팔억 구천칠백사십이만` → A가 0을 하나 빠뜨림)뿐이라, 오버샘플링 덕분이라기보다 학습 편차 범위로 보는 게 맞다.
- test에 1~3자리가 83개뿐이라(1자리 1개, 2자리 8개) 짧은 숫자 정확도는 표본이 작다.
- 오버샘플링으로 4자리 이상 정확도가 떨어지지는 않았다. valid 정확도가 같아(99.99%) 규칙대로 B를 `ko2num.pt`로 저장했다.

## 저장

In [15]:
def save_model(path, model, config, src_vocab, tgt_vocab, metrics=None):
    # 저장 형식 고정 (통합 시 로드)
    torch.save({
        'model': model.state_dict(),
        'config': config,
        'src_lang': src_vocab.lang, 'tgt_lang': tgt_vocab.lang,
        'src_vocab': src_vocab.itos, 'tgt_vocab': tgt_vocab.itos,
        'max_len': MAX_LEN,
        'metrics': metrics or {},
    }, path)
    print(f'saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)')


def load_model(path):
    ckpt = torch.load(path, map_location=DEVICE)
    src_vocab = Vocab.from_itos(ckpt['src_lang'], ckpt['src_vocab'])
    tgt_vocab = Vocab.from_itos(ckpt['tgt_lang'], ckpt['tgt_vocab'])
    m = Seq2Seq(len(src_vocab), len(tgt_vocab), **ckpt['config']).to(DEVICE)
    m.load_state_dict(ckpt['model'])
    m.max_len = ckpt['max_len']
    m.eval()
    return m, src_vocab, tgt_vocab, ckpt


SAVE_PATH = os.path.join(SAVE_DIR, f'{SRC}2{TGT}.pt')
save_model(SAVE_PATH, final_model, CONFIG, src_vocab, tgt_vocab,
           metrics={'test_exact_match': exact_match(final_rows), 'history': final_history,
                    'oversampling': USE_OS,
                    'test_exact_match_base': exact_match(test_rows),
                    'test_exact_match_oversample': exact_match(test_rows_os)})

saved: ../ko2num.pt (6.4 MB)


## 불러와서 테스트 / 사용법

```python
m, sv, tv, ckpt = load_model('../ko2num.pt')
greedy_decode(m, '이만 천오백사십육', sv, tv)   # '21546'
beam_decode(m, '이만 천오백사십육', sv, tv, beam=5)
```
입력은 데이터 파일 표기 그대로의 한글 문자열 ("일만"이 아니라 "만", 만·억 뒤 띄어쓰기), 출력은 숫자 문자열

In [16]:
m, sv, tv, ckpt = load_model(SAVE_PATH)
print(f"loaded {ckpt['src_lang']} -> {ckpt['tgt_lang']} | test acc {ckpt['metrics']['test_exact_match']*100:.2f}%\n")
for s, t, _ in test_loader.dataset.pairs[:5]:
    print(f'입력 {s!r}\n  greedy: {greedy_decode(m, s, sv, tv)}\n  beam  : {beam_decode(m, s, sv, tv)}\n  정답  : {t}')

loaded ko -> num | test acc 100.00%

입력 '구억 천삼백이십육만 구천사백삼십삼'
  greedy: 913269433
  beam  : 913269433
  정답  : 913269433
입력 '육천이백십구만 팔천팔백사십이'
  greedy: 62198842
  beam  : 62198842
  정답  : 62198842
입력 '십사만 사천칠십사'
  greedy: 144074
  beam  : 144074
  정답  : 144074
입력 '삼백십육만 천팔백오십육'
  greedy: 3161856
  beam  : 3161856
  정답  : 3161856
입력 '구천이백십오만 팔백사십일'
  greedy: 92150841
  beam  : 92150841
  정답  : 92150841
